# 🤖 Model Training & Hyperparameter Tuning
## Employee Attrition Prediction

**Objective**: Train multiple classifiers, compare performance, and tune the best model using Optuna.

**Models**:
1. 📐 Logistic Regression (baseline)
2. 🌲 Random Forest
3. 🚀 XGBoost
4. 🔧 Optuna-tuned XGBoost (final)

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import build_pipeline
from src.modeling import (
    train_logistic_regression, train_random_forest, train_xgboost,
    tune_with_optuna, save_model, train_all_models
)
from src.evaluation import get_metrics, classification_report_df
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All modules loaded ✅')

All modules loaded ✅


## 1. Prepare Data

In [2]:
DATA_PATH = '../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv'

pipeline = build_pipeline(
    filepath=DATA_PATH,
    test_size=0.2,
    random_state=42,
    apply_smote=True,
    apply_scaling=True
)

X_train = pipeline['X_train']
X_test = pipeline['X_test']
y_train = pipeline['y_train']
y_test = pipeline['y_test']


Performing Smart Hybrid Feature Selection...
1. Filtering by Correlation (> 0.05): 31 features remain.
2. Checking VIF (removing VIF > 10)...


   Features remaining after VIF filter: 28
3. Running Boruta Feature Selection...


   Boruta selected 10 features (Confirmed + Tentative).
   Boruta selected 10 features. Padding up to 15 features using RF importance.



Final Top 15 Features selected:
  1. Age
  2. DailyRate
  3. StockOptionLevel
  4. TotalWorkingYears
  5. YearsAtCompany
Before SMOTE: {0: 986, 1: 190}
After SMOTE:  {0: 986, 1: 986}

OK: Pipeline complete!
   Training samples: 1972 | Test samples: 294
   Features: 15


## 2. Train Baseline — Logistic Regression

In [3]:
# Baseline model
lr_model = train_logistic_regression(X_train, y_train)

# Evaluate on test set
lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

lr_metrics = get_metrics(y_test, lr_pred, lr_proba)
print('\n📊 Logistic Regression Results:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v:.4f}')

DONE: Logistic Regression trained successfully.

📊 Logistic Regression Results:
  Accuracy: 0.6973
  Precision: 0.3056
  Recall: 0.7021
  F1-Score: 0.4258
  ROC-AUC: 0.7765


## 3. Train Random Forest

In [4]:
rf_model = train_random_forest(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = get_metrics(y_test, rf_pred, rf_proba)
print('\n📊 Random Forest Results:')
for k, v in rf_metrics.items():
    print(f'  {k}: {v:.4f}')

DONE: Random Forest trained successfully.



📊 Random Forest Results:
  Accuracy: 0.8367
  Precision: 0.4872
  Recall: 0.4043
  F1-Score: 0.4419
  ROC-AUC: 0.7797


## 4. Train XGBoost

In [5]:
xgb_model = train_xgboost(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_metrics = get_metrics(y_test, xgb_pred, xgb_proba)
print('\n📊 XGBoost Results:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:.4f}')

DONE: XGBoost trained successfully.

📊 XGBoost Results:
  Accuracy: 0.8469
  Precision: 0.5294
  Recall: 0.3830
  F1-Score: 0.4444
  ROC-AUC: 0.8100


## 5. Quick Comparison (Before Tuning)

In [6]:
# Compare all three models
comparison = pd.DataFrame([
    {'Model': 'Logistic Regression', **lr_metrics},
    {'Model': 'Random Forest', **rf_metrics},
    {'Model': 'XGBoost', **xgb_metrics},
])

print('\n' + '='*70)
print('MODEL COMPARISON (Before Tuning)')
print('='*70)
print(comparison.set_index('Model').round(4).to_string())
print('='*70)

# Highlight: which model has the best Recall?
best_recall = comparison.loc[comparison['Recall'].idxmax()]
print(f"\n🏆 Best Recall: {best_recall['Model']} ({best_recall['Recall']:.4f})")
print('💡 Recall is our priority metric — HR wants to catch employees who will leave!')


MODEL COMPARISON (Before Tuning)
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                              
Logistic Regression    0.6973     0.3056  0.7021    0.4258   0.7765
Random Forest          0.8367     0.4872  0.4043    0.4419   0.7797
XGBoost                0.8469     0.5294  0.3830    0.4444   0.8100

🏆 Best Recall: Logistic Regression (0.7021)
💡 Recall is our priority metric — HR wants to catch employees who will leave!


## 6. Hyperparameter Tuning with Optuna
Tuning Logistic Regression to maximize **Recall** using 5-fold cross-validation.

In [7]:
# Optuna tuning — optimize for Recall
tuning_result = tune_with_optuna(
    model_type='logistic_regression',
    X_train=X_train,
    y_train=y_train,
    n_trials=50,
    random_state=42,
    scoring='recall'
)

best_model = tuning_result['model']
print(f"\nBest CV Recall: {tuning_result['best_score']:.4f}")
print(f"Best Params: {tuning_result['best_params']}")

  0%|          | 0/50 [00:00<?, ?it/s]


[BEST] recall score: 0.8448
PARAMS: Best parameters: {'C': 0.00021714662990815416, 'solver': 'liblinear', 'max_iter': 857, 'penalty': 'l2'}

Best CV Recall: 0.8448
Best Params: {'C': 0.00021714662990815416, 'solver': 'liblinear', 'max_iter': 857, 'penalty': 'l2', 'class_weight': 'balanced', 'random_state': 42}


In [8]:
# Evaluate tuned model on test set
tuned_pred = best_model.predict(X_test)
tuned_proba = best_model.predict_proba(X_test)[:, 1]

tuned_metrics = get_metrics(y_test, tuned_pred, tuned_proba)
print('\n📊 Tuned Logistic Regression Results:')
for k, v in tuned_metrics.items():
    print(f'  {k}: {v:.4f}')

# Final comparison
final_comparison = pd.DataFrame([
    {'Model': 'Logistic Regression (Default)', **lr_metrics},
    {'Model': 'Random Forest', **rf_metrics},
    {'Model': 'XGBoost', **xgb_metrics},
    {'Model': 'Logistic Regression (Tuned) ⭐', **tuned_metrics},
])

print('\n' + '='*70)
print('FINAL MODEL COMPARISON')
print('='*70)
print(final_comparison.set_index('Model').round(4).to_string())
print('='*70)


📊 Tuned Logistic Regression Results:
  Accuracy: 0.5170
  Precision: 0.2189
  Recall: 0.7872
  F1-Score: 0.3426
  ROC-AUC: 0.7358

FINAL MODEL COMPARISON
                               Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                                        
Logistic Regression (Default)    0.6973     0.3056  0.7021    0.4258   0.7765
Random Forest                    0.8367     0.4872  0.4043    0.4419   0.7797
XGBoost                          0.8469     0.5294  0.3830    0.4444   0.8100
Logistic Regression (Tuned) ⭐    0.5170     0.2189  0.7872    0.3426   0.7358


In [9]:
# Save the best model
save_model(best_model, '../models/best_model.pkl')

# Also save the scaler for deployment
import joblib
joblib.dump(pipeline['scaler'], '../models/scaler.pkl')
print('💾 Scaler saved to: ../models/scaler.pkl')

# Save feature names for deployment
import json
with open('../models/feature_names.json', 'w') as f:
    json.dump(pipeline['feature_names'], f)
print('💾 Feature names saved to: ../models/feature_names.json')

 Model saved to: ../models/best_model.pkl
💾 Scaler saved to: ../models/scaler.pkl
💾 Feature names saved to: ../models/feature_names.json


## ✅ Summary

- Trained 3 base models + 1 tuned model
- **Optuna** was used to find optimal Logistic Regression hyperparameters (50 trials)
- Best model saved to `models/best_model.pkl`
- Scaler and feature names also saved for deployment

### 📌 Next: Notebook 04 — Detailed Model Evaluation